In [1]:
# Fix loi: torchao 0.10.0 qua cu (transformers yeu cau >0.16.0)
# Luu y: torchao>=0.16 can torch>=2.6. Sau khi chay xong -> RESTART runtime/kernel roi chay lai.
!pip install -U "torchao>=0.16.0"

KeyboardInterrupt: 

# Assignment 2.5 - Fine-tune Stable Diffusion bang LoRA

Fine-tune SD-1.5 tren **5-10 anh ca nhan tu chup** (KHONG dung nguoi noi tieng).
- Dat anh vao thu muc `instance_images/`
- Train LoRA (chi cap nhat adapter nho, dong bang phan con lai)
- Danh gia **CLIP-I** (do giong subject) va **CLIP-T** (do bam prompt)
- Luu anh sinh ra vao `output/`


In [ ]:
# pip install -q diffusers transformers peft accelerate safetensors
import os, glob, math
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DDIMScheduler
from transformers import CLIPTextModel, CLIPTokenizer
from peft import LoraConfig

OUTPUT_DIR = "output"
INSTANCE_DIR = "Itay"           # <-- bo 5-10 anh cua Itay vao day
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INSTANCE_DIR, exist_ok=True)

model_id = "runwayml/stable-diffusion-v1-5"
device = "cuda" if torch.cuda.is_available() else "cpu"
# train o float32 cho on dinh gradient (LoRA params nho)
weight_dtype = torch.float32

# Subject: nguoi dan ong ten Itay.
# Dung token hiem "sks" lam dinh danh + danh tu lop "man" theo kieu DreamBooth
# de tranh prior cua SD ve cai ten that. Khi sinh anh chi can dung "sks man".
UNIQUE_TOKEN = "sks"   # dinh danh hiem gan voi Itay
CLASS_NOUN = "man"
INSTANCE_PROMPT = f"a photo of {UNIQUE_TOKEN} {CLASS_NOUN}"
print("instance prompt:", INSTANCE_PROMPT)

instance prompt: a photo of sks man


In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device, weight_dtype)
vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae").to(device, weight_dtype)
unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet").to(device, weight_dtype)
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

# dong bang toan bo, chi train LoRA
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# r=16 (thay vi 8) de co du "dung luong" hoc chi tiet khuon mat Itay
lora_config = LoraConfig(
    r=16, lora_alpha=16, init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
)
unet.add_adapter(lora_config)
unet.enable_gradient_checkpointing()

lora_params = [p for p in unet.parameters() if p.requires_grad]
print("so tham so LoRA train:", sum(p.numel() for p in lora_params))

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

so tham so LoRA train: 3188736


In [ ]:
class InstanceDataset(Dataset):
    def __init__(self, img_dir, prompt, size=512):
        self.paths = sorted(
            glob.glob(os.path.join(img_dir, "*.jpg")) +
            glob.glob(os.path.join(img_dir, "*.jpeg")) +
            glob.glob(os.path.join(img_dir, "*.png"))
        )
        assert len(self.paths) >= 1, f"Khong tim thay anh trong {img_dir}"
        self.prompt = prompt
        self.tf = transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),   # -> [-1, 1]
        ])
        self.ids = tokenizer(
            prompt, padding="max_length", truncation=True,
            max_length=tokenizer.model_max_length, return_tensors="pt",
        ).input_ids[0]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return {"pixel_values": self.tf(img), "input_ids": self.ids}

train_ds = InstanceDataset(INSTANCE_DIR, INSTANCE_PROMPT)
# batch_size>=2 de DataParallel co the chia cho nhieu GPU (1 GPU van chay binh thuong)
TRAIN_BATCH = 2
train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH, shuffle=True, drop_last=False)
print("so anh train:", len(train_ds))

AssertionError: Khong tim thay anh trong instance_images

In [ ]:
from torch.optim import AdamW
from diffusers import StableDiffusionPipeline
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import save_file, load_file
import matplotlib.pyplot as plt

LORA_ROOT = os.path.join(OUTPUT_DIR, "lora")
os.makedirs(LORA_ROOT, exist_ok=True)

def save_lora(step):
    """Luu LoRA tai 1 moc step: ban .safetensors cho ComfyUI + ban peft de reload nhanh."""
    d = os.path.join(LORA_ROOT, f"checkpoint-{step}")
    os.makedirs(d, exist_ok=True)
    peft_state = get_peft_model_state_dict(unet)
    save_file(peft_state, os.path.join(d, "lora_peft.safetensors"))      # reload trong notebook
    StableDiffusionPipeline.save_lora_weights(                            # dung cho ComfyUI
        save_directory=d, unet_lora_layers=peft_state, safe_serialization=True)
    print(f"  -> saved {d}")

def list_checkpoints():
    if not os.path.isdir(LORA_ROOT):
        return []
    steps = []
    for name in os.listdir(LORA_ROOT):
        if name.startswith("checkpoint-"):
            try:
                steps.append(int(name.split("-")[1]))
            except ValueError:
                pass
    return sorted(steps)

def load_checkpoint(step):
    """Nap lai trong so LoRA cua 1 checkpoint vao unet (de chon ban infer)."""
    path = os.path.join(LORA_ROOT, f"checkpoint-{step}", "lora_peft.safetensors")
    set_peft_model_state_dict(unet, load_file(path))
    print("loaded", path)

# --- Multi-GPU: DataParallel chia batch cho cac GPU (can TRAIN_BATCH >= so GPU) ---
if torch.cuda.device_count() > 1:
    print(f"Su dung {torch.cuda.device_count()} GPU voi DataParallel")
    train_unet = torch.nn.DataParallel(unet)
else:
    train_unet = unet

max_train_steps = 3000
save_every = 1000          # luu checkpoint moi 1000 step
lr = 1e-4
optimizer = AdamW(lora_params, lr=lr)

loss_steps, loss_values = [], []   # ghi lai loss de ve bieu do

unet.train()
global_step = 0
done = False
while not done:
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device, weight_dtype)
        input_ids = batch["input_ids"].to(device)

        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
            enc_hidden = text_encoder(input_ids)[0]

        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy = noise_scheduler.add_noise(latents, noise, t)

        # return_dict=False -> tra tuple, an toan khi gom ket qua tu DataParallel
        noise_pred = train_unet(noisy, t, encoder_hidden_states=enc_hidden, return_dict=False)[0]
        loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        global_step += 1
        loss_steps.append(global_step)
        loss_values.append(loss.item())
        if global_step % 50 == 0:
            print(f"step {global_step}/{max_train_steps} | loss {loss.item():.4f}")
        if global_step % save_every == 0:
            save_lora(global_step)
        if global_step >= max_train_steps:
            done = True
            break

if global_step % save_every != 0:
    save_lora(global_step)   # luu not ban cuoi neu chua trung moc 1000
print("Train xong. Checkpoint co san:", list_checkpoints())

# --- Ve & luu bieu do train loss vao output/ ---
def moving_avg(x, w=20):
    if len(x) < w:
        return x
    c = np.cumsum(np.insert(x, 0, 0))
    return (c[w:] - c[:-w]) / w

plt.figure(figsize=(8, 4))
plt.plot(loss_steps, loss_values, alpha=0.3, label="loss (raw)")
ma = moving_avg(loss_values, 20)
plt.plot(loss_steps[len(loss_steps) - len(ma):], ma, color="C1", label="loss (MA-20)")
for s in list_checkpoints():
    plt.axvline(s, color="gray", ls="--", lw=0.8)
plt.xlabel("step"); plt.ylabel("MSE loss"); plt.title("LoRA training loss")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
loss_plot_path = os.path.join(OUTPUT_DIR, "train_loss.png")
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
print("saved", loss_plot_path)
plt.show()

In [ ]:
# Chon checkpoint de inference
print("Checkpoint co san (step):", list_checkpoints())

# INFER_CHECKPOINT:
#   None     -> dung trong so dang nam trong bo nho (vua train xong, = ban moi nhat)
#   vd 2000  -> load checkpoint-2000 de infer
#   Co the chay rieng cell nay nhieu lan voi cac moc khac nhau de so sanh.
INFER_CHECKPOINT = None

if INFER_CHECKPOINT is not None:
    load_checkpoint(INFER_CHECKPOINT)
    print("Dang dung checkpoint-", INFER_CHECKPOINT)
else:
    print("Dang dung trong so hien tai trong bo nho (ban moi nhat)")

In [ ]:
# Inference thu cong (DDIM + CFG) voi UNET da gan LoRA (dung checkpoint da chon o cell tren)
ddim = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler")

@torch.no_grad()
def generate(prompt, negative_prompt="", guidance_scale=7.0,
             num_inference_steps=40, seed=0, height=512, width=512):
    unet.eval()
    text_in = tokenizer([prompt], padding="max_length", max_length=77,
                        truncation=True, return_tensors="pt").input_ids.to(device)
    uncond_in = tokenizer([negative_prompt], padding="max_length", max_length=77,
                          return_tensors="pt").input_ids.to(device)
    cond = text_encoder(text_in)[0]
    uncond = text_encoder(uncond_in)[0]
    embeds = torch.cat([uncond, cond])

    g = torch.Generator(device=device).manual_seed(seed)
    latents = torch.randn((1, unet.config.in_channels, height // 8, width // 8),
                          generator=g, device=device, dtype=weight_dtype)
    latents = latents * ddim.init_noise_sigma
    ddim.set_timesteps(num_inference_steps, device=device)
    for t in ddim.timesteps:
        inp = ddim.scale_model_input(torch.cat([latents] * 2), t)
        npred = unet(inp, t, encoder_hidden_states=embeds).sample
        un, co = npred.chunk(2)
        npred = un + guidance_scale * (co - un)
        latents = ddim.step(npred, t, latents).prev_sample
    img = vae.decode(latents / vae.config.scaling_factor).sample
    img = (img / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).float().numpy()[0]
    return Image.fromarray((img * 255).round().astype("uint8"))

# Prompt sinh lai khuon mat ong Itay ("sks man") - nhan manh do sac net & chi tiet khuon mat
neg = ("lowres, blurry, out of focus, deformed face, distorted, disfigured, "
       "extra fingers, bad anatomy, asymmetric eyes, watermark, text, cropped, low quality")
eval_prompts = [
    f"a close-up portrait photo of {UNIQUE_TOKEN} {CLASS_NOUN}, face in sharp focus, "
    f"detailed skin texture, catchlight in eyes, soft studio lighting, 85mm, ultra detailed, high resolution",
    f"a professional headshot of {UNIQUE_TOKEN} {CLASS_NOUN}, looking at the camera, "
    f"sharp focus, natural skin, shallow depth of field, bokeh background, photorealistic",
    f"a cinematic portrait of {UNIQUE_TOKEN} {CLASS_NOUN}, dramatic rim lighting, "
    f"highly detailed face, crisp details, 4k, photorealistic",
    f"a photo of {UNIQUE_TOKEN} {CLASS_NOUN} smiling outdoors, golden hour light, "
    f"detailed eyes, sharp, realistic skin, photorealistic",
]
gen_images = []
for i, p in enumerate(eval_prompts):
    im = generate(p, negative_prompt=neg, guidance_scale=7.0, num_inference_steps=40, seed=100 + i)
    path = os.path.join(OUTPUT_DIR, f"lora_gen_{i}.png")
    im.save(path)
    gen_images.append(im)
    print("saved", path)

## Danh gia: CLIP-I & CLIP-T

- **CLIP-I (subject fidelity):** cosine similarity giua embedding anh sinh ra va
  embedding cac anh that trong `instance_images/` (cang cao = giong subject).
- **CLIP-T (prompt fidelity):** cosine similarity giua embedding anh sinh ra va
  embedding text cua prompt (cang cao = bam prompt).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Goi thang vision_model/text_model roi tu chieu (projection) -> luon dung shape,
# khong phu thuoc hanh vi get_image_features/get_text_features cua tung ban transformers.
@torch.no_grad()
def clip_img_embed(pil_images):
    inp = clip_proc(images=pil_images, return_tensors="pt").to(device)
    pooled = clip_model.vision_model(pixel_values=inp["pixel_values"]).pooler_output  # (B, 768)
    e = clip_model.visual_projection(pooled)                                          # (B, 512)
    return F.normalize(e, dim=-1)

@torch.no_grad()
def clip_txt_embed(texts):
    inp = clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    out = clip_model.text_model(input_ids=inp["input_ids"],
                                attention_mask=inp.get("attention_mask"))
    e = clip_model.text_projection(out.pooler_output)                                 # (B, 512)
    return F.normalize(e, dim=-1)

# anh that lam tham chieu
ref_paths = train_ds.paths
ref_images = [Image.open(p).convert("RGB") for p in ref_paths]
ref_emb = clip_img_embed(ref_images)            # (R, D)
gen_emb = clip_img_embed(gen_images)            # (G, D)
txt_emb = clip_txt_embed(eval_prompts)          # (G, D)

# CLIP-I: moi anh sinh so voi trung binh cac anh that
clip_i = (gen_emb @ ref_emb.mean(0, keepdim=True).T).mean().item()
# CLIP-T: moi anh sinh so voi prompt tuong ung
clip_t = (gen_emb * txt_emb).sum(-1).mean().item()

print(f"CLIP-I (subject fidelity): {clip_i:.4f}")
print(f"CLIP-T (prompt fidelity) : {clip_t:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# So sanh anh sinh tu cac checkpoint -> luu output/checkpoint_comparison.png
# Hang = checkpoint (step), Cot = prompt. Dung cung seed de so sanh cong bang.
ckpts = list_checkpoints()
assert len(ckpts) > 0, "Chua co checkpoint nao - chay cell train truoc."

compare_prompts = [
    f"a close-up portrait photo of {UNIQUE_TOKEN} {CLASS_NOUN}, sharp focus, detailed face, studio lighting",
    f"a professional headshot of {UNIQUE_TOKEN} {CLASS_NOUN}, looking at camera, photorealistic",
    f"a photo of {UNIQUE_TOKEN} {CLASS_NOUN} smiling outdoors, golden hour, detailed eyes",
]
compare_seeds = [100, 101, 102]   # 1 seed co dinh moi cot

n_rows, n_cols = len(ckpts), len(compare_prompts)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
axes = np.atleast_2d(axes)

for r, step in enumerate(ckpts):
    load_checkpoint(step)
    for c, (p, s) in enumerate(zip(compare_prompts, compare_seeds)):
        img = generate(p, negative_prompt=neg, guidance_scale=7.0,
                       num_inference_steps=40, seed=s)
        ax = axes[r, c]
        ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(f"prompt {c+1}", fontsize=10)
        if c == 0:
            ax.set_ylabel(f"step {step}", fontsize=11)

fig.suptitle(f'So sanh checkpoint - subject "{UNIQUE_TOKEN} {CLASS_NOUN}" (Itay)', fontsize=12)
plt.tight_layout()
cmp_path = os.path.join(OUTPUT_DIR, "checkpoint_comparison.png")
fig.savefig(cmp_path, dpi=150, bbox_inches="tight")
print("saved", cmp_path)
plt.show()

# tra unet ve checkpoint da chon o cell INFER_CHECKPOINT (None = ban moi nhat)
if INFER_CHECKPOINT is not None:
    load_checkpoint(INFER_CHECKPOINT)

## ComfyUI - thiet ke workflow inference

Da xuat workflow mau ra **`comfyui_workflow.json`** (Load vao ComfyUI bang nut *Load*).
Graph co ban:

```
CheckpointLoaderSimple(SD1.5) --MODEL/CLIP--> LoraLoader(your lora)
LoraLoader --CLIP--> CLIPTextEncode(positive) / CLIPTextEncode(negative)
LoraLoader --MODEL--> KSampler <-- positive/negative cond, EmptyLatentImage
KSampler --LATENT--> VAEDecode --IMAGE--> SaveImage
```

**Buoc dung:** copy `output/lora/pytorch_lora_weights.safetensors` vao
`ComfyUI/models/loras/` (doi ten cho de nho), va checkpoint SD-1.5 vao
`ComfyUI/models/checkpoints/`.

### Node bo sung de tang CHAT LUONG
- **Upscale 2 buoc (hi-res fix):** `UpscaleModelLoader` + `ImageUpscaleWithModel`
  (vd 4x-UltraSharp), hoac latent upscale roi KSampler lan 2 voi denoise ~0.4.
- **FreeU** node: cai thien chi tiet/độ tuong phan, gan vao MODEL.
- **Negative prompt** tot (\"lowres, blurry, deformed, extra limbs\") giam artifact.

### Node bo sung de tang TOC DO
- **LCM-LoRA + LCMScheduler:** giam con ~4-8 step (set KSampler cfg ~1-2,
  sampler `lcm`). Rat nhanh.
- Hoac **SDXL-Turbo / DPM++ SDE Karras** voi 15-20 step.
- Bat **xformers** / `--fast` khi chay ComfyUI de toi uu attention.
